# BSL в Jupyter: от «Привет, мир!» до проведения ЗУП

Начнём с нескольких BSL-ячеек, затем возьмём данные из настоящей ЗУП, исследуем их в Python и остановим проведение документа.

Конфигурация: ЗУП КОРП 3.1.38.92; дата снимка данных: 01.08.2021. Используйте отдельную демо-копию и соответствующую ей выгрузку исходников. Подготовка окружения — в [README](../README.md). Сеанс закрывается последней ячейкой; при прерывании выполните `runtime.close()`.

## Подготовка

Укажите `PLATFORM_BIN`, `CONNECTION_STRING` и `SOURCE_ROOT` для отдельной копии ЗУП и выгрузки её исходников. Если нужны учётные данные, задайте их локально в `RuntimeConfig`. Эта служебная ячейка один раз открывает сеанс; далее идёт BSL.

In [ ]:
from IPython.display import display
from onec_runtime.config import RuntimeConfig
from onec_runtime.session import ExtensionMode, RuntimeSessionConfig
from onec_runtime_jupyter import InteractiveRuntimeSession

PLATFORM_BIN = r'C:\path\to\1cv8\bin'
CONNECTION_STRING = r'File="C:\path\to\ZUP-demo-copy";'
SOURCE_ROOT = r'C:\path\to\ZUP-source'
EXTENSION_MODE = ExtensionMode.AUTO  # MANUAL для ИБ с заранее установленным расширением.

runtime = InteractiveRuntimeSession.start(
    RuntimeSessionConfig(
        runtime=RuntimeConfig(
            platform_bin=PLATFORM_BIN,
            connection_string=CONNECTION_STRING,
            # При необходимости задайте username и password локально.
        ),
        source_root=SOURCE_ROOT,
        extension_mode=EXTENSION_MODE,
    )
)

## Первая BSL-ячейка

`%%bsl` выполняет содержимое ячейки как BSL в настоящем сеансе 1С. Сообщение появляется прямо под ячейкой.

In [ ]:
%%bsl
Сообщить("Привет, мир!");

## Состояние между ячейками

Сохраним процент повышения. Следующая BSL-ячейка увидит эту переменную.

In [ ]:
%%bsl
ПроцентПовышения = 10;

In [ ]:
%%bsl
Сообщить(ПроцентПовышения);

## Метод прямо в notebook

Функция тоже остаётся доступной в следующих ячейках.

In [ ]:
%%bsl
Функция УвеличитьНаПроцент(Значение)
    Возврат Значение * (1 + ПроцентПовышения / 100);
КонецФункции

In [ ]:
%%bsl
Сообщить(УвеличитьНаПроцент(100000));

In [ ]:
%%bsl
ПроцентПовышения = 20;

In [ ]:
%%bsl
Сообщить(УвеличитьНаПроцент(100000));

Первый вызов выводит 110000, второй — 120000. Функция объявлена раньше, но при повторном вызове читает текущее значение `ПроцентПовышения` из notebook-контекста. Это похоже на работу с внешней переменной функции в Python/Jupyter.

## Та же сессия — внутри ЗУП

Теперь используем типовой `КадровыйУчет.СотрудникиОрганизации`. Дата 01.08.2021 относится к данным демоснимка, а 3.1.38.92 — к версии конфигурации. Список сотрудников затем пригодится и для расчёта ФОТ.

In [ ]:
%%bsl
ДатаФОТ = Дата(2021, 8, 1);
ПараметрыСотрудников = КадровыйУчет.ПараметрыПолученияСотрудниковОрганизацийПоСпискуФизическихЛиц();
ПараметрыСотрудников.НачалоПериода = ДатаФОТ;
ПараметрыСотрудников.ОкончаниеПериода = ДатаФОТ;
ПараметрыСотрудников.РаботникиПоТрудовымДоговорам = Истина;

СписокСотрудников = КадровыйУчет.СотрудникиОрганизации(
    Истина, ПараметрыСотрудников).Скопировать(, "Сотрудник");
СписокСотрудников.Свернуть("Сотрудник");
СписокСотрудников.Колонки.Добавить("Период", Новый ОписаниеТипов("Дата"));
Для Каждого СтрокаСотрудника Из СписокСотрудников Цикл
    СтрокаСотрудника.Период = ДатаФОТ;
КонецЦикла;

Другой типовой метод возвращает кадровые данные и ФОТ для этих сотрудников на ту же дату.

In [ ]:
%%bsl
СотрудникиДляДанных = СписокСотрудников.ВыгрузитьКолонку("Сотрудник");
КадровыеДанные = КадровыйУчет.КадровыеДанныеСотрудников(
    Истина,
    СотрудникиДляДанных,
    "Организация,Подразделение,Должность,ФОТ",
    ДатаФОТ
);

## Из 1С в Python

`КадровыеДанные` пока живёт в 1С; одноимённая Python-переменная — proxy. `to_df()` материализует таблицу в обычный `pandas.DataFrame`.

In [ ]:
# КадровыеДанные пока живёт в 1С; to_df() переносит таблицу в pandas.
df = КадровыеДанные.to_df(refs="presentation")
display(df.head(10))
print("Строк с кадровыми данными:", len(df))

Для ссылок можно выбрать представление, UUID или оба значения. Представление удобно читать, UUID — использовать как идентификатор при сравнении и соединении таблиц.

In [ ]:
df_uuid = КадровыеДанные.to_df(refs="uuid")
df_both = КадровыеДанные.to_df(refs="both")
display(df_uuid[["Сотрудник", "Подразделение"]].head())
display(df_both[["Сотрудник", "Сотрудник__uuid", "Подразделение"]].head())

## Обычный pandas

Сгруппируем ФОТ по подразделению и построим одну диаграмму. Бизнес-данные получила ЗУП; Python нужен для исследования результата.

In [ ]:
import matplotlib.pyplot as plt

by_department = (
    df.groupby("Подразделение", dropna=False)["ФОТ"]
      .sum()
      .sort_values()
)
display(by_department.to_frame("ФОТ"))
fig, ax = plt.subplots(figsize=(9, max(4, len(by_department) * 0.38)))
by_department.map(float).plot.barh(ax=ax)
ax.set_xlabel("ФОТ, ₽")
ax.set_title("Плановый ФОТ по подразделениям на 01.08.2021")
fig.tight_layout()
display(fig)
plt.close(fig)

## Материализуем объект документа

Возьмём проведённый `ПриемНаРаботу` с показателем и начислением оклада у сотрудника из выбранного списка. Служебная ячейка найдёт подходящий документ в демокопии и запомнит исходные значения для финального опыта.

In [ ]:
%%bsl
ЗапросПриемов = Новый Запрос;
ЗапросПриемов.Текст =
    "ВЫБРАТЬ
    |   Прием.Ссылка КАК Ссылка,
    |   Прием.Сотрудник КАК Сотрудник,
    |   СтрокаПоказателя.Показатель КАК Показатель,
    |   СтрокаПоказателя.Показатель.Наименование КАК ИмяПоказателя,
    |   СтрокаПоказателя.Значение КАК Значение,
    |   СтрокаПоказателя.ИдентификаторСтрокиВидаРасчета КАК ИдентификаторСтрокиВидаРасчета
    |ИЗ
    |   Документ.ПриемНаРаботу КАК Прием
    |       ВНУТРЕННЕЕ СОЕДИНЕНИЕ Документ.ПриемНаРаботу.Показатели КАК СтрокаПоказателя
    |       ПО Прием.Ссылка = СтрокаПоказателя.Ссылка
    |ГДЕ
    |   Прием.Проведен
    |   И Прием.НачисленияУтверждены
    |   И НЕ Прием.БронированиеПозиции
    |   И Прием.Сотрудник В (&Сотрудники)
    |   И Прием.Дата <= &ДатаДанных
    |УПОРЯДОЧИТЬ ПО
    |   Прием.Дата УБЫВ";
ЗапросПриемов.УстановитьПараметр("ДатаДанных", КонецДня(ДатаФОТ));
ЗапросПриемов.УстановитьПараметр("Сотрудники", СотрудникиДляДанных);
ВыборкаПриемов = ЗапросПриемов.Выполнить().Выбрать();
СсылкаНаПрием = Неопределено;
Пока ВыборкаПриемов.Следующий() Цикл
    Если СтрНайти(НРег(ВыборкаПриемов.ИмяПоказателя), "оклад") = 0 Тогда
        Продолжить;
    КонецЕсли;
    Если ТипЗнч(ВыборкаПриемов.Значение) <> Тип("Число") Тогда
        Продолжить;
    КонецЕсли;
    Если ВыборкаПриемов.Значение <= 0 Тогда
        Продолжить;
    КонецЕсли;
    Кандидат = ВыборкаПриемов.Ссылка.ПолучитьОбъект();
    Для Каждого СтрокаНачисления Из Кандидат.Начисления Цикл
        Если СтрокаНачисления.ИдентификаторСтрокиВидаРасчета = ВыборкаПриемов.ИдентификаторСтрокиВидаРасчета
            И СтрНайти(НРег(Строка(СтрокаНачисления.Начисление)), "оклад") > 0
            И СтрокаНачисления.Размер > 0 Тогда
            СсылкаНаПрием = ВыборкаПриемов.Ссылка;
            СотрудникДляОпыта = ВыборкаПриемов.Сотрудник;
            ПоказательОклада = ВыборкаПриемов.Показатель;
            ИсходныйОклад = ВыборкаПриемов.Значение;
            НачислениеДляОпыта = СтрокаНачисления.Начисление;
            ИсходныйФОТНачисления = СтрокаНачисления.Размер;
            Прием = Кандидат;
            Прием.ДополнительныеСвойства.Вставить(
                "ОтключитьПроверкуДатыЗапретаИзменения", Истина);
            Прервать;
        КонецЕсли;
    КонецЦикла;
    Если СсылкаНаПрием <> Неопределено Тогда
        Прервать;
    КонецЕсли;
КонецЦикла;
Если СсылкаНаПрием = Неопределено Тогда
    ВызватьИсключение "Не найден проведённый прием с показателем и начислением оклада.";
КонецЕсли;
Сообщить(Строка(СсылкаНаПрием) + ": оклад " + Строка(ИсходныйОклад));

In [ ]:
%%bsl
ПоказателиПриема = Прием.Показатели.Выгрузить();
НачисленияПриема = Прием.Начисления.Выгрузить();

In [ ]:
data = Прием.materialize(refs="presentation")
print(data["Дата"], data["Организация"], data["Сотрудник"])
print("Табличные части:", data.tabular_sections)
display(ПоказателиПриема.head(5).to_df(refs="presentation"))
display(НачисленияПриема.head(5).to_df(refs="presentation"))

`materialize()` работает не только с таблицами: объект документа даёт Python-снимок реквизитов и список табличных частей. Нужную табличную часть можно выгрузить отдельно. Аналогично материализуются структуры, массивы и соответствия.

## Большую таблицу можно читать порциями

Если нужен быстрый просмотр, ограничиваем перенос на стороне 1С.

In [ ]:
display(КадровыеДанные.head(10).to_df(refs="presentation"))

## Notebook в VS Code

Для `%%bsl` доступны подсветка, автодополнение, сигнатуры, hover и переход к определению. Например, из вызова `КадровыйУчет.КадровыеДанныеСотрудников` можно перейти к методу в выгрузке ЗУП. Для статьи здесь уместен скриншот этой ячейки с открытым определением.

## Зафиксируем исходный плановый ФОТ

Типовой метод `ТекущиеДанныеОплатыТрудаСотрудников` вернёт ФОТ до экспериментов. Этот результат пригодится для сравнения после проведения.

In [ ]:
%%bsl
ПлановыйФот = ПлановыеНачисленияСотрудников.ТекущиеДанныеОплатыТрудаСотрудников(
    Неопределено, СписокСотрудников);
ПланФОТ = ПлановыйФот.Скопировать(, "Сотрудник,ФОТ");

In [ ]:
display(ПланФОТ.head(10).to_df(refs="presentation"))

## Hot reload: один короткий опыт

В **локальной копии** выгрузки откройте `CommonModules/ПлановыеНачисленияСотрудников/Ext/Module.bsl`. В запросе метода `ТекущиеДанныеОплатыТрудаСотрудников` добавьте поле `ЗначенияСовокупныхТарифныхСтавок.ФОТ * 1.30 КАК ФОТСоСтраховыми` и сохраните файл. Исходный `ФОТ` остаётся в результате; новое поле показывает условную надбавку 30%. Конфигурация ИБ не изменяется.

<details><summary>Полная версия метода для локальной замены</summary>

```bsl
Функция ТекущиеДанныеОплатыТрудаСотрудников(Ссылка, СотрудникиДаты) Экспорт
    ПараметрыПостроения = ЗарплатаКадрыОбщиеНаборыДанных.ПараметрыПостроенияДляСоздатьВТИмяРегистраСрез();
    ПараметрыПостроения.ФормироватьСПериодичностьДень = Ложь;
    ЗарплатаКадрыОбщиеНаборыДанных.ДобавитьВКоллекциюОтбор(
        ПараметрыПостроения.Отборы, "Регистратор", "<>", Ссылка);
    Запрос = Новый Запрос;
    Запрос.МенеджерВременныхТаблиц = Новый МенеджерВременныхТаблиц;
    ЗарплатаКадрыОбщиеНаборыДанных.СоздатьВТИмяРегистраСрезПоследних(
        "ПлановыйФОТИтоги",
        Запрос.МенеджерВременныхТаблиц,
        Истина,
        ЗарплатаКадрыОбщиеНаборыДанных.ОписаниеФильтраДляСоздатьВТИмяРегистра(
            СотрудникиДаты, "Сотрудник"),
        ПараметрыПостроения);
    Запрос.Текст =
        "ВЫБРАТЬ
        |   ЗначенияСовокупныхТарифныхСтавок.Сотрудник,
        |   ЗначенияСовокупныхТарифныхСтавок.Период,
        |   ЗначенияСовокупныхТарифныхСтавок.СовокупнаяТарифнаяСтавка КАК СовокупнаяТарифнаяСтавка,
        |   ЗначенияСовокупныхТарифныхСтавок.ВидТарифнойСтавки,
        |   ЗначенияСовокупныхТарифныхСтавок.ФОТ КАК ФОТ,
        |   ЗначенияСовокупныхТарифныхСтавок.ФОТ * 1.30 КАК ФОТСоСтраховыми
        |ИЗ
        |   ВТПлановыйФОТИтогиСрезПоследних КАК ЗначенияСовокупныхТарифныхСтавок";
    ЗначенияДанныхОплатыТруда = Запрос.Выполнить().Выгрузить();
    Возврат ЗначенияДанныхОплатыТруда;
КонецФункции
```

</details>

Загрузим изменённый метод в текущий сеанс и повторим прежний вызов.

In [ ]:
RELOAD_MODULE_PATH = r'CommonModules\ПлановыеНачисленияСотрудников\Ext\Module.bsl'
runtime.load_worker_module(RELOAD_MODULE_PATH)

In [ ]:
%%bsl
ПлановыйФотСоСтраховыми = ПлановыеНачисленияСотрудников.ТекущиеДанныеОплатыТрудаСотрудников(
    Неопределено, СписокСотрудников);
ПланСоСтраховыми = ПлановыйФотСоСтраховыми.Скопировать(, "Сотрудник,ФОТ,ФОТСоСтраховыми");

In [ ]:
reloaded = ПланСоСтраховыми.to_df(refs="presentation")
display(reloaded.head(10))
print("ФОТ:", reloaded["ФОТ"].sum(), "₽")
print("ФОТ со страховыми:", reloaded["ФОТСоСтраховыми"].sum(), "₽")

К изменённому коду типового метода вернёмся во второй статье. Теперь остановим проведение документа.

## Capture внутри проведения «Приема на работу»

Выбранный документ уже проведён в демокопии. Повторное проведение из notebook остановим в `РасчетЗарплатыРасширенный` перед обработкой таблиц плановых начислений и значений показателей. Номер строки определяем по локальной выгрузке ЗУП КОРП 3.1.38.92: он может отличаться между форматами выгрузки. Для финального опыта возвращаем повышение к 10%.

In [ ]:
%%bsl
ПроцентПовышения = 10;

In [ ]:
from pathlib import Path

CAPTURE_MODULE_PATH = r'CommonModules\РасчетЗарплатыРасширенный\Ext\Module.bsl'
capture_source = (Path(SOURCE_ROOT) / CAPTURE_MODULE_PATH).read_text(encoding="utf-8-sig")
capture_marker = 'Если СтруктураДанных.Свойство("ДанныеОПлановыхНачислениях") Тогда'
capture_lines = [
    number for number, line in enumerate(capture_source.splitlines(), start=1)
    if line.strip() == capture_marker
]
assert len(capture_lines) == 1
runtime.add_capture_point(CAPTURE_MODULE_PATH, capture_lines[0])

In [ ]:
%%bsl
Прием.Записать(РежимЗаписиДокумента.Проведение);

In [ ]:
capture_status = runtime.status()
assert capture_status.state.value == 'captured'
print('Проведение остановлено:', capture_status.operation_id)

Выполнение документа ещё не завершилось. Через `КонтекстОтладки` видны таблицы именно этого вызова. Для просмотра копируем только нужные столбцы.

In [ ]:
%%bsl
СнимокПоказателей = КонтекстОтладки.СтруктураДанных.ЗначенияПоказателей.Скопировать(
    , "Сотрудник,Показатель,Значение");
СнимокПлановыхНачислений = КонтекстОтладки.СтруктураДанных.ДанныеОПлановыхНачислениях.Скопировать(
    , "Сотрудник,Начисление,Размер");

In [ ]:
display(СнимокПоказателей.head(10).to_df(refs="presentation"))
display(СнимокПлановыхНачислений.head(10).to_df(refs="presentation"))

Теперь вызываем ту же функцию `УвеличитьНаПроцент`, которую объявили в начале notebook. Меняем значение показателя оклада и размер соответствующего планового начисления в живых данных проведения.

In [ ]:
%%bsl
СтрокиОклада = КонтекстОтладки.СтруктураДанных.ЗначенияПоказателей.НайтиСтроки(
    Новый Структура("Сотрудник,Показатель", СотрудникДляОпыта, ПоказательОклада)
);
Если СтрокиОклада.Количество() <> 1 Тогда
    ВызватьИсключение "Ожидалась одна строка показателя оклада.";
КонецЕсли;
СтрокаОклада = СтрокиОклада[0];
Если СтрокаОклада.Значение <> ИсходныйОклад Тогда
    ВызватьИсключение "Оклад изменился после выбора документа.";
КонецЕсли;

СтрокиФОТ = КонтекстОтладки.СтруктураДанных.ДанныеОПлановыхНачислениях.НайтиСтроки(
    Новый Структура("Сотрудник,Начисление", СотрудникДляОпыта, НачислениеДляОпыта)
);
Если СтрокиФОТ.Количество() <> 1 Тогда
    ВызватьИсключение "Ожидалась одна строка планового начисления.";
КонецЕсли;
СтрокаФОТ = СтрокиФОТ[0];
Если СтрокаФОТ.Размер <> ИсходныйФОТНачисления Тогда
    ВызватьИсключение "Плановый ФОТ изменился после выбора документа.";
КонецЕсли;
СтрокаОклада.Значение = УвеличитьНаПроцент(СтрокаОклада.Значение);
СтрокаФОТ.Размер = УвеличитьНаПроцент(СтрокаФОТ.Размер);
Сообщить("Оклад внутри проведения: " + Строка(СтрокаОклада.Значение));

In [ ]:
completed = runtime.resume_capture()
runtime.clear_capture_points()
assert completed.succeeded and completed.state.value == 'completed'
assert completed.operation_id == capture_status.operation_id
print('Продолжилось и завершилось то же проведение')

## Проверяем запись и плановый ФОТ

Сначала прочитаем движения именно этого документа: значение периодического показателя и размер планового начисления. Отдельно посмотрим запись регистра `ПлановыйФОТИтоги`, связанную с тем же приемом.

In [ ]:
%%bsl
ЗапросПроверки = Новый Запрос;
ЗапросПроверки.Текст =
    "ВЫБРАТЬ ПЕРВЫЕ 1
    |   Значения.Показатель КАК Показатель,
    |   Значения.Значение КАК Значение
    |ИЗ
    |   РегистрСведений.ЗначенияПериодическихПоказателейРасчетаЗарплатыСотрудников КАК Значения
    |ГДЕ
    |   Значения.Регистратор = &Прием
    |   И Значения.Сотрудник = &Сотрудник
    |   И Значения.Показатель = &Показатель
    |УПОРЯДОЧИТЬ ПО
    |   Значения.Период УБЫВ";
ЗапросПроверки.УстановитьПараметр("Прием", СсылкаНаПрием);
ЗапросПроверки.УстановитьПараметр("Сотрудник", СотрудникДляОпыта);
ЗапросПроверки.УстановитьПараметр("Показатель", ПоказательОклада);
РезультатПроверки = ЗапросПроверки.Выполнить().Выгрузить();

In [ ]:
from decimal import Decimal

indicator = РезультатПроверки.to_df(refs="presentation")
assert len(indicator) == 1
before_salary = ИсходныйОклад.materialize()
expected_salary = before_salary * Decimal("1.10")
assert indicator.iloc[0]["Значение"] == expected_salary
display(indicator)
print("Показатель оклада:", before_salary, "→", expected_salary)

In [ ]:
%%bsl
ЗапросНачисления = Новый Запрос;
ЗапросНачисления.Текст =
    "ВЫБРАТЬ ПЕРВЫЕ 1
    |   Начисления.Начисление КАК Начисление,
    |   Начисления.Размер КАК Размер
    |ИЗ
    |   РегистрСведений.ПлановыеНачисления КАК Начисления
    |ГДЕ
    |   Начисления.Регистратор = &Прием
    |   И Начисления.Сотрудник = &Сотрудник
    |   И Начисления.Начисление = &Начисление
    |УПОРЯДОЧИТЬ ПО
    |   Начисления.Период УБЫВ";
ЗапросНачисления.УстановитьПараметр("Прием", СсылкаНаПрием);
ЗапросНачисления.УстановитьПараметр("Сотрудник", СотрудникДляОпыта);
ЗапросНачисления.УстановитьПараметр("Начисление", НачислениеДляОпыта);
РезультатНачисления = ЗапросНачисления.Выполнить().Выгрузить();

ЗапросФОТДокумента = Новый Запрос;
ЗапросФОТДокумента.Текст =
    "ВЫБРАТЬ ПЕРВЫЕ 1
    |   ПлановыйФОТ.Период КАК Период,
    |   ПлановыйФОТ.ФОТ КАК ФОТ
    |ИЗ
    |   РегистрСведений.ПлановыйФОТИтоги КАК ПлановыйФОТ
    |ГДЕ
    |   ПлановыйФОТ.РегистраторСобытия = &Прием
    |   И ПлановыйФОТ.Сотрудник = &Сотрудник
    |УПОРЯДОЧИТЬ ПО
    |   ПлановыйФОТ.Период УБЫВ";
ЗапросФОТДокумента.УстановитьПараметр("Прием", СсылкаНаПрием);
ЗапросФОТДокумента.УстановитьПараметр("Сотрудник", СотрудникДляОпыта);
РезультатФОТДокумента = ЗапросФОТДокумента.Выполнить().Выгрузить();

In [ ]:
accrual = РезультатНачисления.to_df(refs="presentation")
expected_accrual = ИсходныйФОТНачисления.materialize() * Decimal("1.10")
assert len(accrual) == 1
assert accrual.iloc[0]["Размер"] == expected_accrual
display(accrual)
print("Плановое начисление:", ИсходныйФОТНачисления.materialize(), "→", expected_accrual)

fot_record = РезультатФОТДокумента.to_df(refs="presentation")
display(fot_record)
print("Плановый ФОТ по документу:",
      fot_record.iloc[0]["ФОТ"] if len(fot_record) else "нет записи")

Типовой метод показывает плановый ФОТ на 01.08.2021. Если после приема были другие кадровые события, этот срез может не измениться от повторного проведения раннего документа.

In [ ]:
%%bsl
ПланПослеПроведения = ПлановыеНачисленияСотрудников.ТекущиеДанныеОплатыТрудаСотрудников(
    Неопределено, СписокСотрудников);
ПланПослеКратко = ПланПослеПроведения.Скопировать(, "Сотрудник,ФОТ");

In [ ]:
employee_uuid = СотрудникДляОпыта.materialize().uuid
before_plan = ПланФОТ.to_df(refs="uuid")
after_plan = ПланПослеКратко.to_df(refs="uuid")
before_value = before_plan.loc[before_plan["Сотрудник"] == employee_uuid, "ФОТ"]
after_value = after_plan.loc[after_plan["Сотрудник"] == employee_uuid, "ФОТ"]
if len(before_value) and len(after_value):
    print("Плановый ФОТ на дату снимка:", before_value.iloc[0], "→", after_value.iloc[0])
else:
    print("В срезе на дату снимка нет строки этого сотрудника; ФОТ документа показан выше.")

## От первой ячейки до живого проведения

Начали с `Сообщить("Привет, мир!")`, а закончили изменением данных внутри остановленного типового проведения и продолжением того же вызова. Во второй статье можно подробно разобрать hot reload, в третьей — capture. Расширенный опыт с двумя точками останова, стеком и временной таблицей — в [03-capture.ipynb](03-capture.ipynb).

## Завершение

In [ ]:
runtime.close()